# Causal Intervention Experiments: Bottleneck vs Specialization Hypothesis

This notebook implements cross-layer activation patching experiments on DistilBERT to test:

1. **Bottleneck Hypothesis**: Layer 5 has cleaner negation representations but this knowledge is underutilized because the model reads from Layer 3
2. **Specialization Hypothesis**: Layer 3 representations are uniquely suited for the sentiment task

## Key Experiments

- **L5→L3 patch**: Take negation activations from Layer 5, inject into Layer 3 position
- **L3→L5 patch**: Reverse direction
- **L3→L3 baseline**: Same-layer patching
- **L5→L5 baseline**: Same-layer patching

## Interpretation

| Result | Interpretation |
|--------|----------------|
| L5→L3 works BEST | Bottleneck hypothesis supported |
| L3→L3 works BEST | Specialization hypothesis supported |
| Both work equally | Distributed representation |
| Neither works well | Probes found correlations, not causal features |


## Section 0: Setup & Configuration


In [ ]:
#@title **Configure Environment** { display-mode: "form" }
import os
import sys

# Check if running in Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Check resources
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Available RAM: {ram_gb:.1f} GB")

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    DEVICE = "cuda"
else:
    print("No GPU detected, using CPU")
    DEVICE = "cpu"

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
print(f"\nConfiguration: Device={DEVICE}, Batch size={BATCH_SIZE}")


In [ ]:
# Mount Google Drive and setup repo (Colab only)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone or pull repository
    !git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
    %cd Negation-Origin-Tracing
    
    DRIVE_PATH = '/content/drive/MyDrive/NOT_results'
else:
    DRIVE_PATH = '../experiments'

OUTPUT_DIR = os.path.join(DRIVE_PATH, 'causal_interventions')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# Install dependencies
%pip install -q torch transformers datasets pandas numpy matplotlib seaborn scipy tqdm

print("Dependencies installed.")


In [ ]:
# Core imports
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, '.')

# Publication-quality settings
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'sans-serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style("whitegrid")

# Color scheme
COLORS = {
    'L5_L3': '#e74c3c',      # Red - key hypothesis
    'L3_L5': '#3498db',      # Blue
    'L3_L3': '#2ecc71',      # Green - baseline
    'L5_L5': '#9b59b6',      # Purple - baseline
    'positive': '#27ae60',   # Green
    'negative': '#c0392b',   # Dark red
    'highlight': '#f39c12',  # Gold
}

print("Imports configured.")


## Section 1: Dataset Loading (neg-150-simple)

Load the `neural-intelligence/neg-150-simple` dataset which contains perfect minimal pairs for negation.


In [ ]:
def load_neg150_simple():
    """
    Load the neg-150-simple dataset from HuggingFace.
    This dataset has minimal negation pairs.
    
    Returns:
        DataFrame with columns: positive, negative, label
    """
    try:
        # Try loading from HuggingFace
        print("Loading neural-intelligence/neg-150-simple from HuggingFace...")
        dataset = load_dataset("neural-intelligence/neg-150-simple")
        
        # Convert to DataFrame
        if 'train' in dataset:
            df = pd.DataFrame(dataset['train'])
        elif 'test' in dataset:
            df = pd.DataFrame(dataset['test'])
        else:
            # Use first available split
            split_name = list(dataset.keys())[0]
            df = pd.DataFrame(dataset[split_name])
        
        print(f"Loaded {len(df)} pairs")
        print(f"Columns: {df.columns.tolist()}")
        return df
        
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Will use fallback SST-2 synthetic pairs.")
        return None


def validate_minimal_pairs(df, positive_col='positive', negative_col='negative'):
    """
    Validate that pairs are minimal (differ only by negation).
    
    Returns:
        DataFrame with validation results
    """
    results = []
    negation_words = {'not', "n't", 'no', 'never', 'nothing', 'nobody', 'nowhere', 'neither'}
    
    for idx, row in df.iterrows():
        pos = str(row[positive_col]).lower().split()
        neg = str(row[negative_col]).lower().split()
        
        # Find words that differ
        pos_set = set(pos)
        neg_set = set(neg)
        
        only_in_neg = neg_set - pos_set
        only_in_pos = pos_set - neg_set
        
        # Check if difference is just negation words
        neg_words_added = only_in_neg & negation_words
        is_minimal = len(neg_words_added) > 0 and len(only_in_pos) <= 1
        
        results.append({
            'idx': idx,
            'is_minimal': is_minimal,
            'neg_words_added': list(neg_words_added),
            'words_removed': list(only_in_pos),
            'len_diff': len(neg) - len(pos)
        })
    
    validation_df = pd.DataFrame(results)
    minimal_count = validation_df['is_minimal'].sum()
    print(f"\nValidation Results:")
    print(f"  Minimal pairs: {minimal_count}/{len(df)} ({100*minimal_count/len(df):.1f}%)")
    print(f"  Average length difference: {validation_df['len_diff'].mean():.2f} words")
    
    return validation_df


# Load dataset
neg150_df = load_neg150_simple()


In [ ]:
# Explore the dataset structure
if neg150_df is not None:
    print("Dataset Info:")
    print(f"  Shape: {neg150_df.shape}")
    print(f"  Columns: {neg150_df.columns.tolist()}")
    print("\nSample pairs:")
    for i in range(min(5, len(neg150_df))):
        row = neg150_df.iloc[i]
        # Handle different column names
        if 'positive' in neg150_df.columns:
            pos_col, neg_col = 'positive', 'negative'
        elif 'sentence1' in neg150_df.columns:
            pos_col, neg_col = 'sentence1', 'sentence2'
        elif 'text' in neg150_df.columns:
            pos_col, neg_col = 'text', 'negated_text'
        else:
            pos_col, neg_col = neg150_df.columns[0], neg150_df.columns[1]
        
        print(f"\n  Pair {i+1}:")
        print(f"    Positive: {row[pos_col][:80]}..." if len(str(row[pos_col])) > 80 else f"    Positive: {row[pos_col]}")
        print(f"    Negative: {row[neg_col][:80]}..." if len(str(row[neg_col])) > 80 else f"    Negative: {row[neg_col]}")


In [ ]:
# Fallback: Create synthetic pairs from SST-2 if neg-150-simple not available
def create_synthetic_pairs_from_sst2(n_pairs=100):
    """
    Create synthetic negation pairs from SST-2 by inserting 'not' before sentiment words.
    
    Returns:
        DataFrame with positive, negative, original_label columns
    """
    print("Creating synthetic pairs from SST-2...")
    
    # Load SST-2
    sst2 = load_dataset("glue", "sst2", split="validation")
    
    # Sentiment words to target
    positive_words = {'good', 'great', 'excellent', 'wonderful', 'amazing', 'fantastic', 
                      'love', 'loved', 'loves', 'like', 'liked', 'likes', 'enjoy', 'enjoyed',
                      'beautiful', 'brilliant', 'best', 'perfect', 'happy', 'fun'}
    negative_words = {'bad', 'terrible', 'awful', 'horrible', 'hate', 'hated', 'hates',
                      'boring', 'dull', 'worst', 'poor', 'disappointing', 'sad', 'ugly'}
    
    pairs = []
    
    for item in sst2:
        sentence = item['sentence']
        label = item['label']
        words = sentence.split()
        
        # Find sentiment words
        target_words = positive_words if label == 1 else negative_words
        
        for i, word in enumerate(words):
            word_lower = word.lower().strip('.,!?"\'')
            if word_lower in target_words:
                # Insert 'not' before the sentiment word
                negated_words = words[:i] + ['not'] + words[i:]
                negated_sentence = ' '.join(negated_words)
                
                pairs.append({
                    'positive': sentence,
                    'negative': negated_sentence,
                    'original_label': label,
                    'modified_word': word_lower
                })
                break
        
        if len(pairs) >= n_pairs:
            break
    
    df = pd.DataFrame(pairs)
    print(f"Created {len(df)} synthetic pairs")
    return df


# Use fallback if needed
if neg150_df is None or len(neg150_df) == 0:
    print("\nUsing SST-2 fallback...")
    pairs_df = create_synthetic_pairs_from_sst2(n_pairs=100)
    POS_COL, NEG_COL = 'positive', 'negative'
else:
    pairs_df = neg150_df
    # Detect column names
    if 'positive' in pairs_df.columns:
        POS_COL, NEG_COL = 'positive', 'negative'
    elif 'sentence1' in pairs_df.columns:
        POS_COL, NEG_COL = 'sentence1', 'sentence2'
    elif 'original' in pairs_df.columns:
        POS_COL, NEG_COL = 'original', 'negated'
    else:
        POS_COL, NEG_COL = pairs_df.columns[0], pairs_df.columns[1]

print(f"\nUsing columns: positive='{POS_COL}', negative='{NEG_COL}'")
print(f"Total pairs available: {len(pairs_df)}")


In [ ]:
# Validate minimal pairs
validation_results = validate_minimal_pairs(pairs_df, POS_COL, NEG_COL)

# Select subset for interventions (50-100 pairs)
N_PAIRS = min(100, len(pairs_df))
intervention_df = pairs_df.head(N_PAIRS).copy()
print(f"\nSelected {N_PAIRS} pairs for intervention experiments")


In [ ]:
# Create PyTorch Dataset for intervention pairs
class NegationPairDataset(Dataset):
    """
    Dataset for negation minimal pairs.
    Returns tokenized positive and negative sentences.
    """
    
    def __init__(self, df, tokenizer, pos_col, neg_col, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.pos_col = pos_col
        self.neg_col = neg_col
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize positive sentence
        pos_encoding = self.tokenizer(
            str(row[self.pos_col]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Tokenize negative sentence
        neg_encoding = self.tokenizer(
            str(row[self.neg_col]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'pos_input_ids': pos_encoding['input_ids'].squeeze(0),
            'pos_attention_mask': pos_encoding['attention_mask'].squeeze(0),
            'neg_input_ids': neg_encoding['input_ids'].squeeze(0),
            'neg_attention_mask': neg_encoding['attention_mask'].squeeze(0),
            'idx': idx
        }


print("NegationPairDataset class defined.")


## Section 2: Cross-Layer Activation Patching

Implement the core intervention mechanism that allows patching activations from one layer into another.


In [ ]:
class CrossLayerActivationPatcher:
    """
    Performs cross-layer activation patching.
    
    Key capability: Extract activations from source_layer of one sentence,
    inject into target_layer of another sentence.
    """
    
    def __init__(self, model, device='cuda'):
        """
        Args:
            model: DistilBERT model (base or classification)
            device: Device to run on
        """
        self.model = model.to(device)
        self.model.eval()
        self.device = device
        self.hook_handles = []
        self.activations = {}
    
    def _get_encoder_layer(self, layer_idx: int):
        """
        Get the transformer layer for hooking.
        Handles both base model and classification model wrappers.
        """
        m = self.model
        
        # DistilBERT base model
        if hasattr(m, 'transformer'):
            return m.transformer.layer[layer_idx]
        # DistilBERT classification model
        if hasattr(m, 'distilbert') and hasattr(m.distilbert, 'transformer'):
            return m.distilbert.transformer.layer[layer_idx]
        # BERT style
        if hasattr(m, 'encoder'):
            return m.encoder.layer[layer_idx]
        if hasattr(m, 'bert') and hasattr(m.bert, 'encoder'):
            return m.bert.encoder.layer[layer_idx]
        
        raise ValueError("Could not find transformer layers in model")
    
    def _clear_hooks(self):
        """Remove all registered hooks."""
        for handle in self.hook_handles:
            handle.remove()
        self.hook_handles.clear()
        self.activations.clear()
    
    def extract_activations(self, input_ids, attention_mask, layer_idx):
        """
        Extract activations from a specific layer.
        
        Args:
            input_ids: Token IDs (batch_size, seq_len)
            attention_mask: Attention mask (batch_size, seq_len)
            layer_idx: Layer to extract from
        
        Returns:
            Tensor of activations (batch_size, seq_len, hidden_size)
        """
        self._clear_hooks()
        
        # Create hook to capture activations
        def capture_hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            self.activations[f'layer_{layer_idx}'] = hidden_states.clone()
        
        # Register hook
        layer = self._get_encoder_layer(layer_idx)
        handle = layer.register_forward_hook(capture_hook)
        self.hook_handles.append(handle)
        
        # Forward pass
        with torch.no_grad():
            _ = self.model(
                input_ids=input_ids.to(self.device),
                attention_mask=attention_mask.to(self.device),
                output_hidden_states=True
            )
        
        # Get captured activations
        activations = self.activations[f'layer_{layer_idx}'].clone()
        
        self._clear_hooks()
        return activations
    
    def patch_and_forward(
        self,
        target_input_ids,
        target_attention_mask,
        source_activations,
        target_layer_idx,
        patch_position=None  # None = patch all positions
    ):
        """
        Run forward pass with patched activations at target layer.
        
        Args:
            target_input_ids: Input IDs for target sentence
            target_attention_mask: Attention mask for target
            source_activations: Activations to inject (from source sentence)
            target_layer_idx: Layer to inject activations into
            patch_position: Token position to patch (None = all)
        
        Returns:
            Model outputs with patched activations
        """
        self._clear_hooks()
        
        # Create patching hook
        def patch_hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            
            if patch_position is not None:
                # Patch specific position only
                hidden_states[:, patch_position, :] = source_activations[:, patch_position, :]
            else:
                # Patch all positions
                # Handle batch size mismatch
                if hidden_states.shape[0] == source_activations.shape[0]:
                    hidden_states = source_activations.clone()
                else:
                    # Broadcast if needed
                    hidden_states = source_activations[:hidden_states.shape[0]].clone()
            
            if isinstance(output, tuple):
                return (hidden_states,) + output[1:]
            return hidden_states
        
        # Register hook
        layer = self._get_encoder_layer(target_layer_idx)
        handle = layer.register_forward_hook(patch_hook)
        self.hook_handles.append(handle)
        
        # Forward pass with patching
        with torch.no_grad():
            outputs = self.model(
                input_ids=target_input_ids.to(self.device),
                attention_mask=target_attention_mask.to(self.device),
                output_hidden_states=True
            )
        
        self._clear_hooks()
        return outputs
    
    def cross_layer_patch(
        self,
        source_input_ids,
        source_attention_mask,
        target_input_ids,
        target_attention_mask,
        source_layer_idx,
        target_layer_idx,
        patch_position=None
    ):
        """
        Full cross-layer patching: extract from source_layer, inject into target_layer.
        
        Args:
            source_input_ids: Source sentence tokens
            source_attention_mask: Source attention mask
            target_input_ids: Target sentence tokens
            target_attention_mask: Target attention mask
            source_layer_idx: Layer to extract activations from
            target_layer_idx: Layer to inject activations into
            patch_position: Token position to patch (None = all)
        
        Returns:
            Model outputs after patching
        """
        # Step 1: Extract activations from source at source_layer
        source_activations = self.extract_activations(
            source_input_ids, source_attention_mask, source_layer_idx
        )
        
        # Step 2: Inject into target at target_layer
        outputs = self.patch_and_forward(
            target_input_ids, target_attention_mask,
            source_activations, target_layer_idx, patch_position
        )
        
        return outputs


print("CrossLayerActivationPatcher class defined.")


In [ ]:
# Load the finetuned sentiment classifier
print("Loading finetuned DistilBERT sentiment classifier...")

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
classifier = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
classifier = classifier.to(DEVICE)
classifier.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"  Layers: {classifier.config.num_hidden_layers}")
print(f"  Hidden size: {classifier.config.hidden_size}")
print(f"  Labels: {classifier.config.id2label}")


In [ ]:
# Create dataset and dataloader
intervention_dataset = NegationPairDataset(
    intervention_df, tokenizer, POS_COL, NEG_COL, max_length=128
)
intervention_loader = DataLoader(
    intervention_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Created dataloader with {len(intervention_dataset)} pairs, batch_size={BATCH_SIZE}")


In [ ]:
# Initialize the patcher
patcher = CrossLayerActivationPatcher(classifier, device=DEVICE)

# Quick test
print("Testing patcher...")
test_batch = next(iter(intervention_loader))

# Test extraction
test_activations = patcher.extract_activations(
    test_batch['neg_input_ids'][:2],
    test_batch['neg_attention_mask'][:2],
    layer_idx=3
)
print(f"  Extracted activations shape: {test_activations.shape}")

# Test cross-layer patching
test_outputs = patcher.cross_layer_patch(
    source_input_ids=test_batch['neg_input_ids'][:2],
    source_attention_mask=test_batch['neg_attention_mask'][:2],
    target_input_ids=test_batch['pos_input_ids'][:2],
    target_attention_mask=test_batch['pos_attention_mask'][:2],
    source_layer_idx=5,
    target_layer_idx=3
)
print(f"  Patched output logits shape: {test_outputs.logits.shape}")
print("Patcher test passed!")


## Section 3: Flip Accuracy Measurement

Run the intervention experiments and measure flip accuracy for each condition.


In [ ]:
def get_predictions(model, input_ids, attention_mask, device):
    """
    Get predictions and probabilities from model.
    
    Returns:
        preds: Predicted labels (0 or 1)
        probs: Softmax probabilities
        logits: Raw logits
    """
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device)
        )
    
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    preds = torch.argmax(logits, dim=-1)
    
    return preds, probs, logits


def run_intervention_experiment(
    patcher,
    dataloader,
    source_layer,
    target_layer,
    device
):
    """
    Run a single intervention experiment.
    
    Patches activations from negated sentence (source) into positive sentence (target)
    and measures if sentiment flips from positive to negative.
    
    Args:
        patcher: CrossLayerActivationPatcher instance
        dataloader: DataLoader with negation pairs
        source_layer: Layer to extract negation activations from
        target_layer: Layer to inject activations into
        device: Device to run on
    
    Returns:
        dict with results
    """
    results = {
        'source_layer': source_layer,
        'target_layer': target_layer,
        'flips': [],           # Did sentiment flip?
        'original_preds': [],  # Original prediction on positive sentence
        'patched_preds': [],   # Prediction after patching
        'original_probs': [],  # Original probability for positive class
        'patched_probs': [],   # Patched probability for positive class
        'logit_deltas': [],    # Change in logits
        'confidence_deltas': [],  # Change in confidence
    }
    
    for batch in tqdm(dataloader, desc=f"L{source_layer}->L{target_layer}"):
        pos_input_ids = batch['pos_input_ids'].to(device)
        pos_attention_mask = batch['pos_attention_mask'].to(device)
        neg_input_ids = batch['neg_input_ids'].to(device)
        neg_attention_mask = batch['neg_attention_mask'].to(device)
        
        # Get original predictions on positive sentences
        orig_preds, orig_probs, orig_logits = get_predictions(
            patcher.model, pos_input_ids, pos_attention_mask, device
        )
        
        # Patch negation activations into positive sentence
        patched_outputs = patcher.cross_layer_patch(
            source_input_ids=neg_input_ids,
            source_attention_mask=neg_attention_mask,
            target_input_ids=pos_input_ids,
            target_attention_mask=pos_attention_mask,
            source_layer_idx=source_layer,
            target_layer_idx=target_layer
        )
        
        patched_logits = patched_outputs.logits
        patched_probs = F.softmax(patched_logits, dim=-1)
        patched_preds = torch.argmax(patched_logits, dim=-1)
        
        # Compute metrics
        # A "flip" is when the prediction changes (ideally from positive to negative)
        flips = (orig_preds != patched_preds).cpu().numpy()
        
        # Confidence for positive class (index 1 in SST-2)
        orig_conf = orig_probs[:, 1].cpu().numpy()
        patched_conf = patched_probs[:, 1].cpu().numpy()
        
        # Logit delta
        logit_delta = (patched_logits - orig_logits).abs().mean(dim=-1).cpu().numpy()
        
        # Store results
        results['flips'].extend(flips.tolist())
        results['original_preds'].extend(orig_preds.cpu().numpy().tolist())
        results['patched_preds'].extend(patched_preds.cpu().numpy().tolist())
        results['original_probs'].extend(orig_conf.tolist())
        results['patched_probs'].extend(patched_conf.tolist())
        results['logit_deltas'].extend(logit_delta.tolist())
        results['confidence_deltas'].extend((orig_conf - patched_conf).tolist())
    
    # Compute summary statistics
    results['flip_accuracy'] = np.mean(results['flips'])
    results['mean_logit_delta'] = np.mean(results['logit_deltas'])
    results['mean_confidence_delta'] = np.mean(results['confidence_deltas'])
    results['n_samples'] = len(results['flips'])
    
    return results


print("Intervention experiment functions defined.")


In [ ]:
# Run the four key intervention conditions
print("="*60)
print("RUNNING INTERVENTION EXPERIMENTS")
print("="*60)

# Define the key conditions
CONDITIONS = [
    {'name': 'L5->L3', 'source': 5, 'target': 3, 'color': COLORS['L5_L3']},
    {'name': 'L3->L5', 'source': 3, 'target': 5, 'color': COLORS['L3_L5']},
    {'name': 'L3->L3', 'source': 3, 'target': 3, 'color': COLORS['L3_L3']},
    {'name': 'L5->L5', 'source': 5, 'target': 5, 'color': COLORS['L5_L5']},
]

all_results = {}

for cond in CONDITIONS:
    print(f"\nRunning {cond['name']} (source={cond['source']}, target={cond['target']})...")
    
    results = run_intervention_experiment(
        patcher=patcher,
        dataloader=intervention_loader,
        source_layer=cond['source'],
        target_layer=cond['target'],
        device=DEVICE
    )
    
    all_results[cond['name']] = results
    
    print(f"  Flip accuracy: {results['flip_accuracy']:.3f}")
    print(f"  Mean logit delta: {results['mean_logit_delta']:.4f}")
    print(f"  Mean confidence delta: {results['mean_confidence_delta']:.4f}")

print("\n" + "="*60)
print("EXPERIMENTS COMPLETE")
print("="*60)


In [ ]:
# Also run full cross-layer matrix for comprehensive analysis
print("\nRunning full 6x6 cross-layer patching matrix...")

cross_layer_matrix = np.zeros((6, 6))
cross_layer_results = {}

for source_layer in tqdm(range(6), desc="Source layers"):
    for target_layer in range(6):
        key = f"L{source_layer}->L{target_layer}"
        
        # Skip if already computed in main conditions
        if key in all_results:
            cross_layer_matrix[source_layer, target_layer] = all_results[key]['flip_accuracy']
            cross_layer_results[key] = all_results[key]
            continue
        
        results = run_intervention_experiment(
            patcher=patcher,
            dataloader=intervention_loader,
            source_layer=source_layer,
            target_layer=target_layer,
            device=DEVICE
        )
        
        cross_layer_matrix[source_layer, target_layer] = results['flip_accuracy']
        cross_layer_results[key] = results

print("\nCross-layer matrix complete.")


## Section 4: Statistical Analysis

Perform statistical tests to determine significance of differences between conditions.


In [ ]:
def bootstrap_ci(data, n_bootstrap=1000, ci=0.95):
    """
    Compute bootstrap confidence interval for the mean.
    
    Returns:
        (mean, lower_ci, upper_ci)
    """
    data = np.array(data)
    n = len(data)
    
    # Bootstrap samples
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        bootstrap_means.append(np.mean(sample))
    
    bootstrap_means = np.array(bootstrap_means)
    
    # Compute CI
    alpha = 1 - ci
    lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    
    return np.mean(data), lower, upper


def cohens_d(group1, group2):
    """
    Compute Cohen's d effect size.
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0
    
    return (np.mean(group1) - np.mean(group2)) / pooled_std


print("Statistical functions defined.")


In [ ]:
# Compute bootstrap CIs for each condition
print("Computing bootstrap confidence intervals...\n")

ci_results = {}

for name, results in all_results.items():
    mean, lower, upper = bootstrap_ci(results['flips'], n_bootstrap=2000)
    ci_results[name] = {
        'mean': mean,
        'ci_lower': lower,
        'ci_upper': upper,
        'ci_width': upper - lower
    }
    print(f"{name}: {mean:.3f} [{lower:.3f}, {upper:.3f}]")

print("\nConfidence intervals computed.")


In [ ]:
# Perform paired t-tests between key conditions
print("Statistical Comparisons (McNemar's Test for Paired Binary Data)")
print("="*60)

comparisons = [
    ('L5->L3', 'L3->L3', 'Bottleneck vs Specialization'),
    ('L5->L3', 'L5->L5', 'L5->L3 vs L5->L5 baseline'),
    ('L3->L3', 'L5->L5', 'L3->L3 vs L5->L5'),
    ('L5->L3', 'L3->L5', 'L5->L3 vs L3->L5'),
]

comparison_results = []

for cond1, cond2, description in comparisons:
    flips1 = np.array(all_results[cond1]['flips'])
    flips2 = np.array(all_results[cond2]['flips'])
    
    # McNemar's test for paired binary data
    # Count discordant pairs
    b = np.sum((flips1 == 1) & (flips2 == 0))  # cond1 flips, cond2 doesn't
    c = np.sum((flips1 == 0) & (flips2 == 1))  # cond2 flips, cond1 doesn't
    
    # McNemar statistic
    if b + c > 0:
        chi2 = (abs(b - c) - 1)**2 / (b + c)  # with continuity correction
        p_value = 1 - stats.chi2.cdf(chi2, df=1)
    else:
        chi2 = 0
        p_value = 1.0
    
    # Effect size
    d = cohens_d(flips1, flips2)
    
    result = {
        'comparison': f"{cond1} vs {cond2}",
        'description': description,
        'mean1': np.mean(flips1),
        'mean2': np.mean(flips2),
        'difference': np.mean(flips1) - np.mean(flips2),
        'chi2': chi2,
        'p_value': p_value,
        'cohens_d': d,
        'significant': p_value < 0.05
    }
    comparison_results.append(result)
    
    sig_marker = "*" if p_value < 0.05 else ""
    print(f"\n{description}:")
    print(f"  {cond1}: {result['mean1']:.3f}")
    print(f"  {cond2}: {result['mean2']:.3f}")
    print(f"  Difference: {result['difference']:+.3f}")
    print(f"  chi2 = {chi2:.3f}, p = {p_value:.4f} {sig_marker}")
    print(f"  Cohen's d = {d:.3f}")

print("\n" + "="*60)


In [ ]:
# Create summary DataFrame
summary_data = []

for name, results in all_results.items():
    ci = ci_results[name]
    summary_data.append({
        'Condition': name,
        'Source Layer': results['source_layer'],
        'Target Layer': results['target_layer'],
        'Flip Accuracy': results['flip_accuracy'],
        'CI Lower': ci['ci_lower'],
        'CI Upper': ci['ci_upper'],
        'Mean Logit Delta': results['mean_logit_delta'],
        'Mean Conf Delta': results['mean_confidence_delta'],
        'N Samples': results['n_samples']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Flip Accuracy', ascending=False)

print("\nResults Summary:")
print(summary_df.to_string(index=False))


## Section 5: Visualization

Create publication-quality visualizations of the results.


In [ ]:
# Figure 1: Bar chart comparing flip accuracy across conditions
fig, ax = plt.subplots(figsize=(10, 6))

conditions = ['L5->L3', 'L3->L3', 'L5->L5', 'L3->L5']
flip_accs = [all_results[c]['flip_accuracy'] for c in conditions]
ci_lowers = [ci_results[c]['ci_lower'] for c in conditions]
ci_uppers = [ci_results[c]['ci_upper'] for c in conditions]
errors = [[fa - cl for fa, cl in zip(flip_accs, ci_lowers)],
          [cu - fa for fa, cu in zip(flip_accs, ci_uppers)]]

colors = [COLORS['L5_L3'], COLORS['L3_L3'], COLORS['L5_L5'], COLORS['L3_L5']]

bars = ax.bar(conditions, flip_accs, yerr=errors, capsize=8, 
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Highlight the key comparison
bars[0].set_edgecolor(COLORS['highlight'])
bars[0].set_linewidth(3)

# Add value labels
for bar, acc in zip(bars, flip_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Intervention Condition', fontsize=14, fontweight='bold')
ax.set_ylabel('Flip Accuracy', fontsize=14, fontweight='bold')
ax.set_title('Cross-Layer Activation Patching: Flip Accuracy by Condition\n(Negation activations patched into positive sentences)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)

# Add hypothesis annotations
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')

# Add legend explaining conditions
legend_text = "L5->L3: Layer 5 negation -> Layer 3 (Bottleneck test)\n" \
              "L3->L3: Layer 3 negation -> Layer 3 (Specialization baseline)"
ax.text(0.02, 0.98, legend_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'flip_accuracy_comparison.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# Figure 2: Cross-layer patching heatmap
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(cross_layer_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Flip Accuracy', fontsize=12, fontweight='bold')

# Add text annotations
for i in range(6):
    for j in range(6):
        value = cross_layer_matrix[i, j]
        text_color = 'white' if value > 0.5 or value < 0.2 else 'black'
        ax.text(j, i, f'{value:.2f}', ha='center', va='center',
                fontsize=10, fontweight='bold', color=text_color)

# Highlight key cells
# L5->L3
rect1 = plt.Rectangle((2.5, 4.5), 1, 1, linewidth=3, edgecolor=COLORS['highlight'], facecolor='none')
ax.add_patch(rect1)
# L3->L3
rect2 = plt.Rectangle((2.5, 2.5), 1, 1, linewidth=3, edgecolor='white', facecolor='none', linestyle='--')
ax.add_patch(rect2)

ax.set_xlabel('Target Layer (injection point)', fontsize=12, fontweight='bold')
ax.set_ylabel('Source Layer (extraction point)', fontsize=12, fontweight='bold')
ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xticklabels([f'L{i}' for i in range(6)])
ax.set_yticklabels([f'L{i}' for i in range(6)])
ax.set_title('Cross-Layer Activation Patching Matrix\n(Gold box: L5->L3 key test, White dashed: L3->L3 baseline)',
             fontsize=14, fontweight='bold')

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'cross_layer_heatmap.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# Figure 3: Confidence delta distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (name, color) in enumerate([(c['name'], c['color']) for c in CONDITIONS]):
    ax = axes[idx // 2, idx % 2]
    
    conf_deltas = all_results[name]['confidence_deltas']
    
    ax.hist(conf_deltas, bins=30, color=color, alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
    ax.axvline(x=np.mean(conf_deltas), color='red', linestyle='-', linewidth=2,
               label=f'Mean: {np.mean(conf_deltas):.3f}')
    
    ax.set_xlabel('Confidence Delta (positive -> after patching)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{name}\nFlip Accuracy: {all_results[name]["flip_accuracy"]:.3f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')

plt.suptitle('Confidence Delta Distributions by Condition\n(Positive delta = reduced confidence in positive class)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'confidence_delta_distributions.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# Figure 4: Layer-wise analysis - which source layers work best for each target?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Best source layer for each target
ax1 = axes[0]
best_sources = np.argmax(cross_layer_matrix, axis=0)
best_values = np.max(cross_layer_matrix, axis=0)

colors = [COLORS['highlight'] if i == 3 else '#3498db' for i in range(6)]
bars = ax1.bar(range(6), best_values, color=colors, alpha=0.8, edgecolor='black')

for i, (bar, src) in enumerate(zip(bars, best_sources)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'L{src}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_xlabel('Target Layer', fontsize=12, fontweight='bold')
ax1.set_ylabel('Best Flip Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Best Source Layer for Each Target\n(Label shows optimal source)',
              fontsize=12, fontweight='bold')
ax1.set_xticks(range(6))
ax1.set_xticklabels([f'L{i}' for i in range(6)])

# Panel 2: Diagonal vs off-diagonal comparison
ax2 = axes[1]
diagonal = np.diag(cross_layer_matrix)
off_diagonal_max = [np.max(np.delete(cross_layer_matrix[:, i], i)) for i in range(6)]

x = np.arange(6)
width = 0.35

bars1 = ax2.bar(x - width/2, diagonal, width, label='Same-layer (diagonal)',
                color='#2ecc71', alpha=0.8, edgecolor='black')
bars2 = ax2.bar(x + width/2, off_diagonal_max, width, label='Best cross-layer',
                color='#e74c3c', alpha=0.8, edgecolor='black')

ax2.set_xlabel('Layer', fontsize=12, fontweight='bold')
ax2.set_ylabel('Flip Accuracy', fontsize=12, fontweight='bold')
ax2.set_title('Same-Layer vs Cross-Layer Patching\n(Does cross-layer patching help?)',
              fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'L{i}' for i in range(6)])
ax2.legend()

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'layer_analysis.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## Section 6: Interpretation Framework

Analyze results and determine which hypothesis is supported.


In [ ]:
def interpret_results(all_results, comparison_results):
    """
    Interpret the intervention results and determine hypothesis support.
    
    Returns:
        dict with interpretation
    """
    interpretation = {
        'bottleneck_support': 0,
        'specialization_support': 0,
        'distributed_support': 0,
        'evidence': [],
        'conclusion': ''
    }
    
    # Get key metrics
    l5_l3 = all_results['L5->L3']['flip_accuracy']
    l3_l3 = all_results['L3->L3']['flip_accuracy']
    l5_l5 = all_results['L5->L5']['flip_accuracy']
    l3_l5 = all_results['L3->L5']['flip_accuracy']
    
    # Find the key comparison result
    key_comparison = next((c for c in comparison_results 
                          if c['comparison'] == 'L5->L3 vs L3->L3'), None)
    
    # Evidence 1: L5->L3 vs L3->L3
    if l5_l3 > l3_l3:
        interpretation['bottleneck_support'] += 2
        interpretation['evidence'].append(
            f"L5->L3 ({l5_l3:.3f}) > L3->L3 ({l3_l3:.3f}): Supports BOTTLENECK - "
            "Layer 5 negation knowledge is more effective when injected into Layer 3"
        )
    elif l3_l3 > l5_l3:
        interpretation['specialization_support'] += 2
        interpretation['evidence'].append(
            f"L3->L3 ({l3_l3:.3f}) > L5->L3 ({l5_l3:.3f}): Supports SPECIALIZATION - "
            "Layer 3's own representations are better suited for the task"
        )
    else:
        interpretation['distributed_support'] += 1
        interpretation['evidence'].append(
            f"L5->L3 = L3->L3: Suggests DISTRIBUTED representation"
        )
    
    # Evidence 2: Statistical significance
    if key_comparison and key_comparison['significant']:
        if key_comparison['difference'] > 0:
            interpretation['bottleneck_support'] += 1
            interpretation['evidence'].append(
                f"Statistically significant (p={key_comparison['p_value']:.4f}): "
                "L5->L3 advantage is reliable"
            )
        else:
            interpretation['specialization_support'] += 1
            interpretation['evidence'].append(
                f"Statistically significant (p={key_comparison['p_value']:.4f}): "
                "L3->L3 advantage is reliable"
            )
    else:
        interpretation['distributed_support'] += 1
        interpretation['evidence'].append(
            "No significant difference: Conditions may be equivalent"
        )
    
    # Evidence 3: Baseline comparisons
    if l5_l3 > l5_l5:
        interpretation['bottleneck_support'] += 1
        interpretation['evidence'].append(
            f"L5->L3 ({l5_l3:.3f}) > L5->L5 ({l5_l5:.3f}): "
            "Layer 3 is a better injection point than Layer 5"
        )
    
    if l3_l3 > l3_l5:
        interpretation['specialization_support'] += 1
        interpretation['evidence'].append(
            f"L3->L3 ({l3_l3:.3f}) > L3->L5 ({l3_l5:.3f}): "
            "Layer 3 representations work best at Layer 3"
        )
    
    # Evidence 4: Overall flip rates
    max_flip = max(l5_l3, l3_l3, l5_l5, l3_l5)
    if max_flip < 0.3:
        interpretation['evidence'].append(
            f"WARNING: Low overall flip rates (max={max_flip:.3f}). "
            "Probes may have found correlations, not causal features."
        )
    
    # Determine conclusion
    scores = {
        'BOTTLENECK': interpretation['bottleneck_support'],
        'SPECIALIZATION': interpretation['specialization_support'],
        'DISTRIBUTED': interpretation['distributed_support']
    }
    
    winner = max(scores, key=scores.get)
    interpretation['conclusion'] = winner
    interpretation['scores'] = scores
    
    return interpretation


# Run interpretation
interpretation = interpret_results(all_results, comparison_results)

print("="*70)
print("INTERPRETATION OF RESULTS")
print("="*70)

print("\nEvidence:")
for i, evidence in enumerate(interpretation['evidence'], 1):
    print(f"  {i}. {evidence}")

print(f"\nHypothesis Scores:")
for hyp, score in interpretation['scores'].items():
    print(f"  {hyp}: {score}")

print(f"\n{'='*70}")
print(f"CONCLUSION: {interpretation['conclusion']} HYPOTHESIS")
print(f"{'='*70}")


In [ ]:
# Create interpretation summary figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Hypothesis support scores
ax1 = axes[0]
hypotheses = list(interpretation['scores'].keys())
scores = list(interpretation['scores'].values())
colors = ['#e74c3c', '#2ecc71', '#3498db']

bars = ax1.bar(hypotheses, scores, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

# Highlight winner
winner_idx = hypotheses.index(interpretation['conclusion'])
bars[winner_idx].set_edgecolor(COLORS['highlight'])
bars[winner_idx].set_linewidth(4)

ax1.set_xlabel('Hypothesis', fontsize=12, fontweight='bold')
ax1.set_ylabel('Evidence Score', fontsize=12, fontweight='bold')
ax1.set_title('Hypothesis Support Based on Intervention Results',
              fontsize=12, fontweight='bold')

# Panel 2: Key comparison visualization
ax2 = axes[1]

# Create comparison table
table_data = [
    ['L5->L3 (Bottleneck test)', f"{all_results['L5->L3']['flip_accuracy']:.3f}"],
    ['L3->L3 (Specialization baseline)', f"{all_results['L3->L3']['flip_accuracy']:.3f}"],
    ['Difference', f"{all_results['L5->L3']['flip_accuracy'] - all_results['L3->L3']['flip_accuracy']:+.3f}"],
]

# Find significance
key_comp = next((c for c in comparison_results if c['comparison'] == 'L5->L3 vs L3->L3'), None)
if key_comp:
    table_data.append(['p-value', f"{key_comp['p_value']:.4f}"])
    table_data.append(['Significant?', 'Yes' if key_comp['significant'] else 'No'])

ax2.axis('off')
table = ax2.table(
    cellText=table_data,
    colLabels=['Metric', 'Value'],
    loc='center',
    cellLoc='center',
    colWidths=[0.6, 0.3]
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2)

# Color header
for i in range(2):
    table[(0, i)].set_facecolor('#34495e')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax2.set_title('Key Comparison: Bottleneck vs Specialization',
              fontsize=12, fontweight='bold', pad=20)

plt.suptitle(f'Conclusion: {interpretation["conclusion"]} Hypothesis Supported',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'interpretation_summary.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## Section 7: Save Results


In [ ]:
# Compile all results
final_results = {
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'name': 'neg-150-simple' if neg150_df is not None else 'sst2-synthetic',
        'n_pairs': N_PAIRS,
        'positive_column': POS_COL,
        'negative_column': NEG_COL
    },
    'model': MODEL_NAME,
    'device': DEVICE,
    'conditions': {
        name: {
            'source_layer': results['source_layer'],
            'target_layer': results['target_layer'],
            'flip_accuracy': results['flip_accuracy'],
            'mean_logit_delta': results['mean_logit_delta'],
            'mean_confidence_delta': results['mean_confidence_delta'],
            'n_samples': results['n_samples'],
            'ci_lower': ci_results[name]['ci_lower'],
            'ci_upper': ci_results[name]['ci_upper']
        }
        for name, results in all_results.items()
    },
    'cross_layer_matrix': cross_layer_matrix.tolist(),
    'statistical_comparisons': comparison_results,
    'interpretation': {
        'conclusion': interpretation['conclusion'],
        'scores': interpretation['scores'],
        'evidence': interpretation['evidence']
    }
}

# Save to JSON
results_path = os.path.join(OUTPUT_DIR, 'causal_intervention_results.json')
with open(results_path, 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print(f"Results saved to: {results_path}")

# Save summary DataFrame
summary_path = os.path.join(OUTPUT_DIR, 'intervention_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")


In [ ]:
# Final summary
print("\n" + "="*70)
print("CAUSAL INTERVENTION EXPERIMENTS COMPLETE")
print("="*70)

print(f"\nDataset: {final_results['dataset']['name']} ({N_PAIRS} pairs)")
print(f"Model: {MODEL_NAME}")

print("\nKey Results:")
print(f"  L5->L3 (Bottleneck test):    {all_results['L5->L3']['flip_accuracy']:.3f}")
print(f"  L3->L3 (Specialization):     {all_results['L3->L3']['flip_accuracy']:.3f}")
print(f"  L5->L5 (Baseline):           {all_results['L5->L5']['flip_accuracy']:.3f}")
print(f"  L3->L5 (Reverse):            {all_results['L3->L5']['flip_accuracy']:.3f}")

print(f"\nConclusion: {interpretation['conclusion']} hypothesis is best supported")

print(f"\nOutput files saved to: {OUTPUT_DIR}")
print("  - causal_intervention_results.json")
print("  - intervention_summary.csv")
print("  - flip_accuracy_comparison.png")
print("  - cross_layer_heatmap.png")
print("  - confidence_delta_distributions.png")
print("  - layer_analysis.png")
print("  - interpretation_summary.png")

print("\n" + "="*70)
